# RAIKYO - AI Sales Consultant Mitsubishi

RAIKYO adalah sistem RAG (Retrieval-Augmented Generation) yang berfungsi sebagai AI Sales Consultant untuk produk Mitsubishi Indonesia.
Sistem ini mampu membaca brosur PDF, memahami konten teks dan gambar, lalu menjawab pertanyaan pelanggan secara akurat berbasis data.

Alur pipeline:
1. Ingestion - membaca semua PDF dari folder dataset
2. Parsing - ekstrak teks dan gambar menggunakan OpenAI Vision API
3. Chunking - memecah dokumen menjadi potongan yang optimal
4. Embedding dan Vector Store - simpan ke Pinecone
5. Hybrid Search - gabungan semantic dan keyword search
6. Generation - jawaban berbasis konteks dokumen
7. Evaluasi - mengukur kualitas sistem RAG

Langkah 1 - Instalasi Library

In [6]:
!pip install -q langchain langchain-openai langchain-pinecone langchain-community pinecone-client python-dotenv pypdf pymupdf openai pillow rank_bm25 tiktoken tqdm langchain-text-splitters langchain-core

Langkah 2 - Import dan Konfigurasi

In [7]:
!pip install -q langchain-text-splitters langchain-core

In [9]:
import os
import base64
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

import fitz  # PyMuPDF
from PIL import Image
import io

from dotenv import load_dotenv
from openai import OpenAI

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate


from pinecone import Pinecone, ServerlessSpec
from rank_bm25 import BM25Okapi

import warnings
warnings.filterwarnings('ignore')

load_dotenv()

print("Semua library berhasil diimport")

Semua library berhasil diimport


In [10]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV", "us-east-1")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY tidak ditemukan di file .env")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY tidak ditemukan di file .env")

INDEX_NAME = "raikyo-mitsubishi-sales"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
CHAT_MODEL = "gpt-4o-mini"
DATASET_FOLDER = "./dataset"

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("Konfigurasi berhasil dimuat")
print(f"OpenAI API Key: {'*' * 20}{OPENAI_API_KEY[-4:]}")
print(f"Pinecone API Key: {'*' * 20}{PINECONE_API_KEY[-4:]}")
print(f"Index Name: {INDEX_NAME}")
print(f"Dataset Folder: {DATASET_FOLDER}")

Konfigurasi berhasil dimuat
OpenAI API Key: ********************CQ4A
Pinecone API Key: ********************YW7A
Index Name: raikyo-mitsubishi-sales
Dataset Folder: ./dataset


Langkah 3 - Inisialisasi Pinecone Index

In [11]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Membuat index baru: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Index {INDEX_NAME} berhasil dibuat")
else:
    print(f"Index {INDEX_NAME} sudah ada, melanjutkan proses")

pinecone_index = pc.Index(INDEX_NAME)
print(f"\nStatus index:")
print(pinecone_index.describe_index_stats())

Membuat index baru: raikyo-mitsubishi-sales
Index raikyo-mitsubishi-sales berhasil dibuat

Status index:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


Langkah 4 - Parsing PDF dengan OpenAI Vision API


In [ ]:
def extract_page_as_image_base64(pdf_path: str, page_num: int, dpi: int = 150) -> str:
    
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    img_bytes = pix.tobytes("png")
    doc.close()
    return base64.b64encode(img_bytes).decode("utf-8")


def parse_page_with_vision(pdf_path: str, page_num: int, source_name: str) -> Dict[str, Any]:
    
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    raw_text = page.get_text("text").strip()
    doc.close()

    img_b64 = extract_page_as_image_base64(pdf_path, page_num)

    prompt = (
        "Kamu adalah asisten ekstraksi data untuk brosur kendaraan Mitsubishi.\n"
        "Ekstrak SEMUA informasi dari halaman ini secara lengkap dan terstruktur.\n"
        "Fokus pada: nama model, spesifikasi teknis (mesin, transmisi, dimensi, berat, kapasitas), "
        "fitur unggulan, harga jika ada, keunggulan produk, dan target pengguna.\n"
        "Jika ada tabel spesifikasi, tuliskan semua baris dan nilainya.\n"
        "Jawab dalam Bahasa Indonesia yang jelas dan terstruktur."
    )

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{img_b64}",
                                "detail": "high"
                            }
                        }
                    ]
                }
            ],
            max_tokens=1500
        )
        vision_text = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Vision API error pada {source_name} halaman {page_num}: {e}")
        vision_text = ""

    combined = raw_text
    if vision_text and len(vision_text) > len(raw_text) * 0.5:
        combined = f"{raw_text}\n\n[Analisis Visual]:\n{vision_text}"
    elif vision_text:
        combined = f"{vision_text}\n\n[Teks Asli]:\n{raw_text}"

    return {
        "content": combined,
        "metadata": {
            "source": source_name,
            "page": page_num + 1,
            "type": "pdf_page",
            "has_vision": bool(vision_text)
        }
    }


print("Fungsi parser PDF siap digunakan")

Fungsi parser PDF siap digunakan


Langkah 5 - Data Ingestion (Load Semua PDF)

In [13]:
def load_all_pdfs(folder_path: str, use_vision: bool = True, max_files: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Load pymu dan open ai.
    """
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder {folder_path} tidak ditemukan.")

    pdf_files = sorted(folder.glob("*.pdf"))
    if not pdf_files:
        raise ValueError(f"Tidak ada file PDF di {folder_path}")

    if max_files:
        pdf_files = pdf_files[:max_files]

    print(f"Ditemukan {len(pdf_files)} file PDF")
    all_documents = []

    for file_idx, pdf_file in enumerate(pdf_files, 1):
        print(f"\n[{file_idx}/{len(pdf_files)}] Memproses: {pdf_file.name}")

        try:
            doc = fitz.open(str(pdf_file))
            total_pages = len(doc)
            doc.close()

            for page_num in range(total_pages):
                if use_vision:
                    page_data = parse_page_with_vision(str(pdf_file), page_num, pdf_file.name)
                else:
                    doc = fitz.open(str(pdf_file))
                    raw_text = doc[page_num].get_text("text").strip()
                    doc.close()
                    page_data = {
                        "content": raw_text,
                        "metadata": {
                            "source": pdf_file.name,
                            "page": page_num + 1,
                            "type": "pdf_page",
                            "has_vision": False
                        }
                    }

                if page_data["content"].strip():
                    all_documents.append(page_data)

            print(f"  Selesai: {total_pages} halaman diproses")

        except Exception as e:
            print(f"  Error pada {pdf_file.name}: {e}")
            continue

    print(f"\nTotal dokumen berhasil diload: {len(all_documents)}")
    return all_documents


documents = load_all_pdfs(
    DATASET_FOLDER,
    use_vision=True
)

print(f"\nContoh konten dokumen pertama:")
print(documents[0]["content"][:500])
print(f"\nMetadata: {documents[0]['metadata']}")

Ditemukan 19 file PDF

[1/19] Memproses: mitsubishi-colt-fe-71-911094.pdf
  Selesai: 2 halaman diproses

[2/19] Memproses: mitsubishi-colt-fe-71-l-240729.pdf
  Selesai: 2 halaman diproses

[3/19] Memproses: mitsubishi-colt-fe-73-528533.pdf
  Selesai: 2 halaman diproses

[4/19] Memproses: mitsubishi-colt-fe-74-hd-539730.pdf
  Selesai: 2 halaman diproses

[5/19] Memproses: mitsubishi-colt-fe-74-l-104152.pdf
  Selesai: 2 halaman diproses

[6/19] Memproses: mitsubishi-colt-fe-shdx-477280.pdf
  Selesai: 2 halaman diproses

[7/19] Memproses: mitsubishi-destinator-352097 (1).pdf
  Selesai: 7 halaman diproses

[8/19] Memproses: mitsubishi-destinator-352097.pdf
  Selesai: 7 halaman diproses

[9/19] Memproses: mitsubishi-ecanter-243699.pdf
  Selesai: 11 halaman diproses

[10/19] Memproses: mitsubishi-fe-74-hds-564374.pdf
  Selesai: 2 halaman diproses

[11/19] Memproses: mitsubishi-fe-84gs-638464.pdf
  Selesai: 2 halaman diproses

[12/19] Memproses: mitsubishi-l100-ev-394594.pdf
  Selesai: 4 hala

Langkah 6 - Chunking Dokumen

In [14]:
def chunk_documents(documents: List[Dict[str, Any]]) -> List[Document]:
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []

    for doc in documents:
        texts = text_splitter.split_text(doc["content"])

        for chunk_idx, text in enumerate(texts):
            if text.strip():
                langchain_doc = Document(
                    page_content=text,
                    metadata={
                        **doc["metadata"],
                        "chunk_id": chunk_idx
                    }
                )
                all_chunks.append(langchain_doc)

    print(f"Total chunks yang dihasilkan: {len(all_chunks)}")

    sources = set(chunk.metadata["source"] for chunk in all_chunks)
    print(f"Jumlah sumber dokumen: {len(sources)}")

    return all_chunks


chunks = chunk_documents(documents)

print(f"\nContoh chunk pertama:")
print(f"Konten: {chunks[0].page_content[:300]}")
print(f"Metadata: {chunks[0].metadata}")

Total chunks yang dihasilkan: 231
Jumlah sumber dokumen: 19

Contoh chunk pertama:
Konten: [Analisis Visual]:
Berikut adalah ekstraksi informasi dari brosur kendaraan Mitsubishi Fuso:

### Nama Model
- **Mitsubishi Fuso FE 71 PS**

### Spesifikasi Teknis
- **Mesin**: Colt Diesel
- **Transmisi**: Manual (spesifikasi lebih lanjut tidak tercantum)
- **Dimensi**: (spesifikasi lebih lanjut tid
Metadata: {'source': 'mitsubishi-colt-fe-71-911094.pdf', 'page': 1, 'type': 'pdf_page', 'has_vision': True, 'chunk_id': 0}


Langkah 7 - Embedding dan Penyimpanan ke Pinecone

In [15]:
def create_vector_store(chunks: List[Document]) -> PineconeVectorStore:
    
    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        openai_api_key=OPENAI_API_KEY
    )

    print(f"Membuat embeddings untuk {len(chunks)} chunks...")

    BATCH_SIZE = 100
    total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    vector_store = None

    for batch_idx in range(total_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(chunks))
        batch = chunks[start:end]

        print(f"Batch {batch_idx + 1}/{total_batches}: chunk {start + 1} - {end}")

        try:
            if vector_store is None:
                vector_store = PineconeVectorStore.from_documents(
                    documents=batch,
                    embedding=embeddings,
                    index_name=INDEX_NAME
                )
            else:
                vector_store.add_documents(batch)

            time.sleep(0.5)

        except Exception as e:
            print(f"Error pada batch {batch_idx + 1}: {e}")
            time.sleep(2)
            continue

    print(f"\nSemua chunks berhasil: {INDEX_NAME}")
    return vector_store


vector_store = create_vector_store(chunks)

print("\nVerifikasi isi index:")
print(pinecone_index.describe_index_stats())

Membuat embeddings untuk 231 chunks...
Batch 1/3: chunk 1 - 100
Batch 2/3: chunk 101 - 200
Batch 3/3: chunk 201 - 231

Semua chunks berhasil: raikyo-mitsubishi-sales

Verifikasi isi index:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 231}},
 'total_vector_count': 231,
 'vector_type': 'dense'}


Langkah 8 - Hybrid Search (Semantic + BM25 Keyword Search)

In [16]:
class HybridSearchEngine:
    

    def __init__(self, vector_store: PineconeVectorStore, chunks: List[Document]):
        self.vector_store = vector_store
        self.chunks = chunks

        tokenized_corpus = [doc.page_content.lower().split() for doc in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

        print("Hybrid Search Engine siap")
        print(f"BM25 corpus: {len(chunks)} dokumen")

    def semantic_search(self, query: str, k: int = 10) -> List[Document]:
        results = self.vector_store.similarity_search(query, k=k)
        return results

    def bm25_search(self, query: str, k: int = 10) -> List[Document]:
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )[:k]

        return [self.chunks[i] for i in top_indices]

    def hybrid_search(self, query: str, k: int = 5, alpha: float = 0.6) -> List[Document]:
        
        semantic_results = self.semantic_search(query, k=k * 2)
        bm25_results = self.bm25_search(query, k=k * 2)

        doc_scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}

        RRF_K = 60

        for rank, doc in enumerate(semantic_results):
            doc_id = f"{doc.metadata.get('source', '')}_{doc.metadata.get('page', '')}_{doc.metadata.get('chunk_id', '')}_{doc.page_content[:50]}"
            score = alpha * (1 / (RRF_K + rank + 1))
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + score
            doc_map[doc_id] = doc

        for rank, doc in enumerate(bm25_results):
            doc_id = f"{doc.metadata.get('source', '')}_{doc.metadata.get('page', '')}_{doc.metadata.get('chunk_id', '')}_{doc.page_content[:50]}"
            score = (1 - alpha) * (1 / (RRF_K + rank + 1))
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + score
            doc_map[doc_id] = doc

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_map[doc_id] for doc_id, _ in sorted_docs[:k]]


hybrid_engine = HybridSearchEngine(vector_store, chunks)

Hybrid Search Engine siap
BM25 corpus: 231 dokumen


Langkah 9 - Pipeline RAG dengan LangChain

In [ ]:
RAIKYO_PROMPT_TEMPLATE = """
Kamu adalah RAIKYO, AI Sales Consultant profesional untuk produk Mitsubishi Indonesia.
Tugasmu adalah membantu pelanggan menemukan kendaraan Mitsubishi yang paling sesuai dengan kebutuhan mereka.

Panduan menjawab:
1. Jawab berdasarkan informasi dari konteks yang diberikan.
2. Jika ada beberapa pilihan yang relevan, bandingkan secara objektif.
3. Berikan rekomendasi yang spesifik dan beralasan.
4. Sebutkan spesifikasi teknis yang relevan dengan pertanyaan.
5. Gunakan bahasa yang ramah, profesional, dan mudah dipahami.
6. Jika informasi tidak tersedia di konteks, katakan dengan jujur bahwa informasi tidak tersedia.

Konteks dari brosur Mitsubishi:
{context}

Pertanyaan pelanggan: {question}

Jawaban RAIKYO:"""


class RAIKYOSystem:


    def __init__(self, hybrid_engine: HybridSearchEngine):
        self.hybrid_engine = hybrid_engine
        self.llm = ChatOpenAI(
            model=CHAT_MODEL,
            temperature=0.3,
            openai_api_key=OPENAI_API_KEY
        )
        self.prompt = PromptTemplate(
            template=RAIKYO_PROMPT_TEMPLATE,
            input_variables=["context", "question"]
        )
        self.conversation_history: List[Dict[str, str]] = []
        print("RAIKYO System siap melayani pelanggan")

    def retrieve_context(self, query: str, k: int = 5) -> str:
        
        relevant_docs = self.hybrid_engine.hybrid_search(query, k=k)

        context_parts = []
        for i, doc in enumerate(relevant_docs, 1):
            source = doc.metadata.get("source", "Unknown")
            page = doc.metadata.get("page", "?")
            context_parts.append(
                f"[Sumber {i}: {source}, Halaman {page}]\n{doc.page_content}"
            )

        return "\n\n".join(context_parts)

    def query(self, question: str, k: int = 5) -> Dict[str, Any]:
        
        context = self.retrieve_context(question, k=k)

        formatted_prompt = self.prompt.format(
            context=context,
            question=question
        )

        response = self.llm.invoke(formatted_prompt)
        answer = response.content

        self.conversation_history.append({
            "question": question,
            "answer": answer
        })

        return {
            "question": question,
            "answer": answer,
            "context": context
        }

    def chat(self, question: str) -> str:
        
        result = self.query(question)
        return result["answer"]


raikyo = RAIKYOSystem(hybrid_engine)
print("RAIKYO siap menerima pertanyaan")

RAIKYO System siap melayani pelanggan
RAIKYO siap menerima pertanyaan


Langkah 10 - Evaluasi Sistem RAG

In [18]:
def evaluate_rag_system(raikyo_system: RAIKYOSystem) -> Dict[str, Any]:
    
    eval_questions = [
        {
            "question": "Apa spesifikasi mesin Mitsubishi Colt Diesel FE 71?",
            "expected_keywords": ["4D34", "mesin", "diesel", "kapasitas", "daya"]
        },
        {
            "question": "Saya punya usaha logistik pengiriman antar kota, kendaraan Mitsubishi mana yang paling cocok?",
            "expected_keywords": ["niaga", "kapasitas", "daya angkut", "rekomendasi"]
        },
        {
            "question": "Apa perbedaan Mitsubishi Xpander dan Xpander Cross?",
            "expected_keywords": ["Xpander", "Cross", "perbedaan", "fitur"]
        },
        {
            "question": "Berapa kapasitas bahan bakar Mitsubishi Pajero Sport?",
            "expected_keywords": ["Pajero", "tangki", "liter", "bahan bakar"]
        },
        {
            "question": "Kendaraan listrik apa yang ada di lineup Mitsubishi?",
            "expected_keywords": ["listrik", "electric", "eCanter", "EV"]
        }
    ]

    print("=" * 70)
    print("EVALUASI SISTEM RAIKYO")
    print("=" * 70)

    results = []

    for i, eval_item in enumerate(eval_questions, 1):
        print(f"\nEval {i}/{len(eval_questions)}: {eval_item['question']}")
        print("-" * 50)

        result = raikyo_system.query(eval_item["question"])

        answer_lower = result["answer"].lower()
        context_lower = result["context"].lower()

        keyword_hits_answer = sum(
            1 for kw in eval_item["expected_keywords"]
            if kw.lower() in answer_lower
        )
        keyword_hits_context = sum(
            1 for kw in eval_item["expected_keywords"]
            if kw.lower() in context_lower
        )

        total_keywords = len(eval_item["expected_keywords"])
        answer_relevance = keyword_hits_answer / total_keywords
        context_relevance = keyword_hits_context / total_keywords

        eval_result = {
            "question": eval_item["question"],
            "answer": result["answer"],
            "answer_relevance_score": round(answer_relevance, 2),
            "context_relevance_score": round(context_relevance, 2),
            "keyword_hits_answer": keyword_hits_answer,
            "keyword_hits_context": keyword_hits_context,
            "total_keywords": total_keywords
        }
        results.append(eval_result)

        print(f"Jawaban: {result['answer'][:300]}...")
        print(f"Relevansi Jawaban: {answer_relevance:.0%} ({keyword_hits_answer}/{total_keywords} keyword ditemukan)")
        print(f"Relevansi Konteks: {context_relevance:.0%} ({keyword_hits_context}/{total_keywords} keyword di konteks)")

    avg_answer_relevance = sum(r["answer_relevance_score"] for r in results) / len(results)
    avg_context_relevance = sum(r["context_relevance_score"] for r in results) / len(results)

    print("\n" + "=" * 70)
    print("RINGKASAN EVALUASI")
    print("=" * 70)
    print(f"Rata-rata Relevansi Jawaban : {avg_answer_relevance:.0%}")
    print(f"Rata-rata Relevansi Konteks : {avg_context_relevance:.0%}")
    print(f"Jumlah pertanyaan dievaluasi: {len(results)}")

    return {
        "results": results,
        "avg_answer_relevance": avg_answer_relevance,
        "avg_context_relevance": avg_context_relevance
    }


eval_results = evaluate_rag_system(raikyo)

EVALUASI SISTEM RAIKYO

Eval 1/5: Apa spesifikasi mesin Mitsubishi Colt Diesel FE 71?
--------------------------------------------------
Jawaban: Spesifikasi mesin dari Mitsubishi Colt Diesel FE 71 adalah sebagai berikut:

- **Model Mesin:** 4034-2AT5
- **Tipe:** 4 silinder, diesel
- **Jumlah Silinder:** 4
- **Diameter x Langkah:** 104 mm x 115 mm
- **Isi Silinder:** 3.908 cc
- **Daya Maksimum:** 90 Ps pada RPM tertentu
- **Torsi Maksimum:** ...
Relevansi Jawaban: 80% (4/5 keyword ditemukan)
Relevansi Konteks: 100% (5/5 keyword di konteks)

Eval 2/5: Saya punya usaha logistik pengiriman antar kota, kendaraan Mitsubishi mana yang paling cocok?
--------------------------------------------------
Jawaban: Untuk usaha logistik pengiriman antar kota, saya merekomendasikan **Mitsubishi Fuso FE 74 L** sebagai pilihan yang paling cocok. Berikut adalah beberapa alasan dan perbandingan dengan model lainnya:

### Mitsubishi Fuso FE 74 L
- **Daya**: 125 PS, memberikan tenaga yang cukup untuk mengan

Langkah 11 - Testing End-to-End

In [19]:
test_scenarios = [
    {
        "scenario": "Pelanggan perusahaan logistik",
        "question": "Perusahaan kami butuh armada truk untuk pengiriman barang 5 ton kapasitas, Mitsubishi apa yang cocok?"
    },
    {
        "scenario": "Keluarga mencari MPV",
        "question": "Saya keluarga dengan 3 anak, butuh mobil yang nyaman dan irit. Mitsubishi apa yang direkomendasikan?"
    },
    {
        "scenario": "Pertanyaan teknis spesifik",
        "question": "Apa sistem transmisi yang digunakan di Mitsubishi Triton 2024?"
    },
    {
        "scenario": "Perbandingan produk",
        "question": "Apa perbedaan antara Mitsubishi Destinator dan eCanter untuk kebutuhan kendaraan komersial?"
    },
    {
        "scenario": "Kendaraan ramah lingkungan",
        "question": "Saya ingin beralih ke kendaraan lebih ramah lingkungan untuk bisnis. Apakah Mitsubishi punya pilihan?"
    }
]

print("=" * 70)
print("TESTING END-TO-END RAIKYO SALES CONSULTANT")
print("=" * 70)

for i, test in enumerate(test_scenarios, 1):
    print(f"\nTEST {i}/{len(test_scenarios)}: {test['scenario']}")
    print("-" * 50)
    print(f"Pertanyaan: {test['question']}")
    print()

    try:
        jawaban = raikyo.chat(test["question"])
        print(f"RAIKYO: {jawaban}")
    except Exception as e:
        print(f"Error: {e}")

    print("-" * 50)

TESTING END-TO-END RAIKYO SALES CONSULTANT

TEST 1/5: Pelanggan perusahaan logistik
--------------------------------------------------
Pertanyaan: Perusahaan kami butuh armada truk untuk pengiriman barang 5 ton kapasitas, Mitsubishi apa yang cocok?

RAIKYO: Untuk kebutuhan armada truk pengiriman barang dengan kapasitas 5 ton, saya merekomendasikan **Mitsubishi Fuso FE 74 L** dan **Mitsubishi Fuso FE 74 HD**. Kedua model ini memiliki spesifikasi yang sesuai untuk kebutuhan bisnis Anda.

### 1. Mitsubishi Fuso FE 74 L
- **Mesin**: Mitsubishi 4D34, Daya 125 PS
- **Transmisi**: Manual 5-percepatan
- **Fitur Unggulan**: Super Capacity, 6 Ban, desain yang kuat dan tahan banting, serta efisiensi bahan bakar yang baik.
- **Target Pengguna**: Cocok untuk pelaku usaha kecil dan menengah, perusahaan logistik, dan individu yang membutuhkan kendaraan angkut berkapasitas besar.

### 2. Mitsubishi Fuso FE 74 HD
- **Mesin**: Diesel, Daya 125 PS
- **Transmisi**: Manual
- **Fitur Unggulan**: Super power

Langkah 12 - Mode Chat Interaktif

In [1]:
def interactive_chat():
    
    print("=" * 70)
    print("SELAMAT DATANG DI RAIKYO - AI Sales Consultant Mitsubishi")
    print("Silakan tanyakan apa saja tentang produk Mitsubishi Indonesia")
    print("Ketik 'exit' atau 'keluar' untuk mengakhiri sesi")
    print("=" * 70)

    while True:
        try:
            question = input("\nAnda: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSesi chat diakhiri")
            break

        if not question:
            continue

        if question.lower() in ["exit", "quit", "keluar", "selesai"]:
            print("\nTerima kasih telah menggunakan RAIKYO! Semoga menemukan kendaraan yang tepat.")
            break

        try:
            jawaban = raikyo.chat(question)
            print(f"\nRAIKYO: {jawaban}")
        except Exception as e:
            print(f"\nError: {e}")


print("Untuk memulai sesi chat interaktif, jalankan perintah berikut:")
print("interactive_chat()")

Untuk memulai sesi chat interaktif, jalankan perintah berikut:
interactive_chat()
